# CycleGAN: Unpaired Sketch ↔ Photo Domain Adaptation
### Question 3 — Domain Adaptation & Unpaired Image-to-Image Translation

| | |
|---|---|
| **Model** | CycleGAN |
| **Domain A** | Sketch (TU-Berlin + Sketchy + QuickDraw) |
| **Domain B** | Photo (Sketchy photo subset) |
| **Tasks** | Sketch→Photo AND Photo→Sketch |
| **Platform** | Kaggle T4×2 Dual GPU |

---
### Dataset Structure Notes
- **Sketchy** (`sharanyasundar/sketchy-dataset`): has `photo/tx_000000000000/` (real photos) and `sketch/tx_000000000000/` (sketches) organised by category subfolders
- **QuickDraw** (`quickdraw-doodle-recognition`): `train_simplified/` contains 340 CSV files, one per category; each row has a `drawing` column with JSON stroke data that we render to images
- **TU-Berlin** (HuggingFace `sdiaeyu6n/tu-berlin`): downloaded via `datasets` library; images are PIL objects in the `image` column
- **For CycleGAN**: we need UNPAIRED Domain A (sketches) and Domain B (photos) — no matching required

## Cell 1 — Install Dependencies

In [1]:
!pip install -q kagglehub datasets scikit-image gradio torchvision Pillow

## Cell 2 — Imports & Device Setup

In [2]:
import os, gc, glob, io, json, random, shutil, time, warnings, zipfile
import itertools
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from tqdm.notebook import tqdm
from skimage.metrics import structural_similarity as compute_ssim
from skimage.metrics import peak_signal_noise_ratio as compute_psnr

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torch.amp import GradScaler, autocast

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPUS = torch.cuda.device_count()

def unwrap(m): return m.module if hasattr(m, 'module') else m

print(f'Device : {device}  |  GPUs: {N_GPUS}')
print(f'PyTorch: {torch.__version__}')
for i in range(N_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  ({p.total_memory//1024**2} MB)')

Device : cuda  |  GPUs: 1
PyTorch: 2.10.0+cu128
  GPU 0: Tesla T4  (14912 MB)


## Cell 3 — Global Configuration

In [3]:
IMG_SIZE       = 128      # 128x128 as specified (fits dual T4 with batch=4)
BATCH_SIZE     = 4        # 4-8 per spec; 4 safer for 128px + CycleGAN memory
NUM_EPOCHS     = 20       # CycleGAN needs more epochs than Pix2Pix
LR             = 2e-4
BETAS          = (0.5, 0.999)
N_RESBLOCKS    = 6        # 6 ResNet blocks (spec says 6 for Kaggle)
LAMBDA_CYCLE   = 10.0     # cycle consistency weight
LAMBDA_ID      = 5.0      # identity loss weight (half of cycle)
DECAY_START    = 25       # start LR decay at this epoch
SAVE_EVERY     = 10
NUM_WORKERS    = 2

# Subset sizes to fit Kaggle session time
SUBSET_SKETCH  = 3000     # Domain A: sketches
SUBSET_PHOTO   = 3000     # Domain B: photos
BUFFER_SIZE    = 50       # replay buffer for discriminator

CKPT_DIR  = '/kaggle/working/ckpt_cyclegan'
OUT_DIR   = '/kaggle/working/outputs'
HF_DIR    = '/kaggle/working/hf_deploy'
for d in [CKPT_DIR, OUT_DIR, HF_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'IMG_SIZE={IMG_SIZE} | BATCH={BATCH_SIZE} | EPOCHS={NUM_EPOCHS}')
print(f'LAMBDA_CYCLE={LAMBDA_CYCLE} | LAMBDA_ID={LAMBDA_ID}')
print(f'N_RESBLOCKS={N_RESBLOCKS}')

IMG_SIZE=128 | BATCH=4 | EPOCHS=20
LAMBDA_CYCLE=10.0 | LAMBDA_ID=5.0
N_RESBLOCKS=6


## Cell 4 — Download Datasets

In [4]:
import kagglehub, zipfile, tarfile, os, glob

# ── Utility: extract any zip/tar inside a directory ───────────────
def extract_archives(root):
    """
    Recursively finds and extracts ALL zip/tar archives inside root.
    This is needed because kagglehub sometimes downloads zips without extracting.
    """
    extracted_any = False
    for dirpath, _, files in os.walk(root):
        for fname in files:
            fpath = os.path.join(dirpath, fname)
            # ZIP
            if fname.endswith('.zip'):
                out_dir = os.path.join(dirpath, os.path.splitext(fname)[0])
                if not os.path.exists(out_dir):
                    print(f'  Extracting zip: {fname} -> {out_dir}/')
                    os.makedirs(out_dir, exist_ok=True)
                    try:
                        with zipfile.ZipFile(fpath, 'r') as zf:
                            zf.extractall(out_dir)
                        extracted_any = True
                    except Exception as e:
                        print(f'    Zip error: {e}')
                else:
                    print(f'  Already extracted: {fname}')
            # TAR
            elif fname.endswith(('.tar.gz', '.tar', '.tgz')):
                out_dir = os.path.join(dirpath, fname.split('.tar')[0])
                if not os.path.exists(out_dir):
                    print(f'  Extracting tar: {fname}')
                    os.makedirs(out_dir, exist_ok=True)
                    try:
                        with tarfile.open(fpath, 'r:*') as tf:
                            tf.extractall(out_dir)
                        extracted_any = True
                    except Exception as e:
                        print(f'    Tar error: {e}')
    return extracted_any


def full_tree(root, max_depth=4, max_files=5):
    """
    Print the FULL directory tree so we know exactly what's inside.
    This is the most important debugging step.
    """
    if not root or not os.path.exists(root):
        print(f'  [NOT FOUND: {root}]'); return
    for dirpath, dirs, files in os.walk(root):
        depth = dirpath.replace(root, '').count(os.sep)
        if depth > max_depth: continue
        indent = '  ' * depth
        rel = os.path.relpath(dirpath, root)
        print(f'{indent}[DIR] {rel}/')
        for f in sorted(files)[:max_files]:
            size_kb = os.path.getsize(os.path.join(dirpath, f)) // 1024
            print(f'{indent}  {f}  ({size_kb} KB)')
        if len(files) > max_files:
            print(f'{indent}  ... and {len(files)-max_files} more files')
        dirs[:] = sorted(dirs)[:6]   # show max 6 subdirs


# ════════════════════════════════════════════════════════════
#  1. SKETCHY DATASET
# ════════════════════════════════════════════════════════════
print('Downloading Sketchy dataset...')
SKETCHY_ROOT = kagglehub.dataset_download('sharanyasundar/sketchy-dataset')
print(f'Sketchy download path: {SKETCHY_ROOT}')
print('\nExtracting archives in Sketchy...')
extract_archives(SKETCHY_ROOT)
print('\nFull Sketchy tree:')
full_tree(SKETCHY_ROOT)

# ════════════════════════════════════════════════════════════
#  2. QUICKDRAW — Competition download or fallback dataset
# ════════════════════════════════════════════════════════════
QUICKDRAW_ROOT = None
print('\n\nDownloading QuickDraw...')
try:
    QUICKDRAW_ROOT = kagglehub.competition_download('quickdraw-doodle-recognition')
    print(f'QuickDraw competition path: {QUICKDRAW_ROOT}')
    full_tree(QUICKDRAW_ROOT, max_depth=2)
except Exception as e:
    print(f'Competition download failed: {e}')
    # Fallback: use a public QuickDraw PNG dataset (no competition acceptance needed)
    print('Trying QuickDraw PNG fallback dataset...')
    try:
        QUICKDRAW_ROOT = kagglehub.dataset_download('yingwurenjian/quickdraw')
        print(f'QuickDraw fallback path: {QUICKDRAW_ROOT}')
        full_tree(QUICKDRAW_ROOT, max_depth=2)
    except Exception as e2:
        print(f'Fallback also failed: {e2}. QuickDraw will be skipped.')
        QUICKDRAW_ROOT = None

# ════════════════════════════════════════════════════════════
#  3. TU-BERLIN (HuggingFace)
# ════════════════════════════════════════════════════════════
print('\n\nLoading TU-Berlin from HuggingFace...')
TUBERLIN_LOADED = False
tuberlin_ds = None
try:
    from datasets import load_dataset
    tuberlin_ds = load_dataset('sdiaeyu6n/tu-berlin', split='train',
                               trust_remote_code=True)
    print(f'TU-Berlin: {len(tuberlin_ds)} samples | columns: {tuberlin_ds.column_names}')
    TUBERLIN_LOADED = True
except Exception as e:
    print(f'TU-Berlin failed: {e}')

print('\nAll downloads done.')

100%|██████████| 12.6M/12.6M [00:01<00:00, 6.68MB/s]

Extracting files...


Sketchy download path: /root/.cache/kagglehub/datasets/sharanyasundar/sketchy-dataset/versions/1

Extracting archives in Sketchy...

Full Sketchy tree:
[DIR] ./
  LICENSE  (1 KB)
  README.md  (7 KB)
  SketchyDataset_README.md  (2 KB)
  retrieval_test.py  (3 KB)
  test_img.txt  (32 KB)
  ... and 2 more files
  [DIR] check/
    __init__.py  (0 KB)
    feature_vis.py  (0 KB)
  [DIR] data/
    __init__.py  (1 KB)
    image_input.py  (1 KB)
    triplet_input.py  (2 KB)
  [DIR] feature/
    feature_extract.py  (2 KB)
  [DIR] models/
    TripletEmbedding.py  (7 KB)
    TripletLoss.py  (1 KB)
    __init__.py  (0 KB)
    sketch_resnet.py  (5 KB)
    vgg.py  (2 KB)
  [DIR] record/
    [DIR] record/feature_vis/
      resnet50_0.png  (57 KB)
      resnet50_315.png  (31 KB)
      resnet50_64_265.png  (41 KB)
      resnet50_940.png  (35 KB)
      resnet_150.png  (57 KB)
      ... and 7 more files
    [DIR] record/retrieval_reslut/
      result.png  (12308 KB)
  [DIR] utils/
    __init__.py  (0 KB)
 

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'sdiaeyu6n/tu-berlin' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'sdiaeyu6n/tu-berlin' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md: 0.00B [00:00, ?B/s]

...

data/test-00000-of-00001.parquet:   0%|          | 0.00/59.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

TU-Berlin: 16000 samples | columns: ['image', 'label']

All downloads done.


## Cell 5 — Explore Dataset Structure

In [5]:
from PIL import Image, ImageDraw
import json

IMG_EXTS = ('*.jpg','*.jpeg','*.png','*.bmp','*.PNG','*.JPG','*.JPEG')

def collect_images(folder, recursive=True):
    """Collect all image files under folder."""
    paths = []
    for ext in IMG_EXTS:
        if recursive:
            paths += glob.glob(os.path.join(folder, '**', ext), recursive=True)
        else:
            paths += glob.glob(os.path.join(folder, ext))
    return sorted([p for p in paths if os.path.getsize(p) > 100])


def find_folders_by_keyword(root, keywords, exclude_keywords=None):
    """
    Walk the tree and return all directory paths whose name contains
    any of the keywords (case-insensitive).
    Exclude dirs containing exclude_keywords.
    """
    found = []
    exclude_keywords = exclude_keywords or []
    for dirpath, dirs, files in os.walk(root):
        name = os.path.basename(dirpath).lower()
        if any(k in name for k in keywords):
            if not any(ex in name for ex in exclude_keywords):
                found.append(dirpath)
    return found


# ════════════════════════════════════════════════════════════
#  DOMAIN B: PHOTOS  — collect from Sketchy
# ════════════════════════════════════════════════════════════
print('=== Collecting Domain B: PHOTOS ===')

# Strategy: find any folder whose name contains 'photo', 'image', 'real', 'img'
# but NOT 'sketch', 'draw', 'line'
photo_dirs = find_folders_by_keyword(
    SKETCHY_ROOT,
    keywords=['photo', 'image', 'real', 'img'],
    exclude_keywords=['sketch', 'draw', 'line', 'info', 'invalid']
)
print(f'Photo-like dirs found: {photo_dirs}')

photo_paths_raw = []
for d in photo_dirs:
    imgs = collect_images(d)
    print(f'  {d}: {len(imgs)} images')
    photo_paths_raw.extend(imgs)

# Deduplicate
photo_paths_raw = sorted(set(photo_paths_raw))
print(f'Total photos from Sketchy: {len(photo_paths_raw)}')

# FALLBACK: if still no photos, grab a general photo dataset
FALLBACK_PHOTO_ROOT = None
if len(photo_paths_raw) < 100:
    print('\nFew/no photos found in Sketchy. Downloading fallback photo dataset...')
    try:
        # Animals-10: small, general, high quality photos
        FALLBACK_PHOTO_ROOT = kagglehub.dataset_download('alessiocorrado99/animals10')
        print(f'Fallback photos path: {FALLBACK_PHOTO_ROOT}')
        extract_archives(FALLBACK_PHOTO_ROOT)
        full_tree(FALLBACK_PHOTO_ROOT, max_depth=2)
        photo_paths_raw = collect_images(FALLBACK_PHOTO_ROOT)
        print(f'Fallback photos found: {len(photo_paths_raw)}')
    except Exception as e:
        print(f'Fallback failed: {e}')
        # Last resort: use TU-Berlin converted to colour as pseudo-photos
        # (only if nothing else works)
        print('Will use TU-Berlin as both domains (sketch-only training).')


# ════════════════════════════════════════════════════════════
#  DOMAIN A: SKETCHES  — Sketchy + TU-Berlin + QuickDraw
# ════════════════════════════════════════════════════════════
print('\n=== Collecting Domain A: SKETCHES ===')

# From Sketchy: find sketch folders
sketch_dirs = find_folders_by_keyword(
    SKETCHY_ROOT,
    keywords=['sketch', 'draw', 'line'],
    exclude_keywords=['photo', 'real', 'img', 'info', 'invalid']
)
print(f'Sketch-like dirs in Sketchy: {sketch_dirs}')

sketchy_sketch_paths = []
for d in sketch_dirs:
    imgs = collect_images(d)
    print(f'  {d}: {len(imgs)} images')
    sketchy_sketch_paths.extend(imgs)
sketchy_sketch_paths = sorted(set(sketchy_sketch_paths))
print(f'Sketchy sketches: {len(sketchy_sketch_paths)}')


# ── TU-Berlin: save to disk ───────────────────────────────────────
TUBERLIN_DIR = '/kaggle/working/tuberlin_sketches'
os.makedirs(TUBERLIN_DIR, exist_ok=True)
tuberlin_sketch_paths = []

if TUBERLIN_LOADED and tuberlin_ds is not None:
    from tqdm.notebook import tqdm
    import numpy as np
    print('\nSaving TU-Berlin sketches...')

    # Auto-detect image column
    img_col = None
    for col in tuberlin_ds.column_names:
        sample_val = tuberlin_ds[0][col]
        if isinstance(sample_val, Image.Image) or hasattr(sample_val, 'mode'):
            img_col = col; break
        try:
            arr = np.array(sample_val)
            if arr.ndim >= 2:
                img_col = col; break
        except Exception:
            continue
    if img_col is None:
        img_col = tuberlin_ds.column_names[0]
    print(f'  Using image column: "{img_col}"')

    max_save = min(2000, len(tuberlin_ds))
    for i in tqdm(range(max_save), desc='TU-Berlin'):
        try:
            val = tuberlin_ds[i][img_col]
            if isinstance(val, Image.Image):
                pil = val
            elif isinstance(val, dict) and 'bytes' in val:
                import io
                pil = Image.open(io.BytesIO(val['bytes']))
            else:
                pil = Image.fromarray(np.array(val).astype(np.uint8))
            out_p = os.path.join(TUBERLIN_DIR, f'tb_{i:05d}.png')
            pil.convert('RGB').save(out_p)
            tuberlin_sketch_paths.append(out_p)
        except Exception:
            continue
    print(f'TU-Berlin saved: {len(tuberlin_sketch_paths)}')


# ── QuickDraw: render from CSV strokes ────────────────────────────
QUICKDRAW_DIR = '/kaggle/working/quickdraw_sketches'
os.makedirs(QUICKDRAW_DIR, exist_ok=True)
quickdraw_sketch_paths = []

def render_quickdraw_strokes(drawing_data, size=128):
    img  = Image.new('RGB', (size, size), 'white')
    draw = ImageDraw.Draw(img)
    try:
        strokes = json.loads(drawing_data) if isinstance(drawing_data, str) else drawing_data
        for stroke in strokes:
            xs, ys = stroke[0], stroke[1]
            pts = [(int(x * size / 256), int(y * size / 256)) for x, y in zip(xs, ys)]
            if len(pts) >= 2:
                draw.line(pts, fill='black', width=2)
    except Exception:
        pass
    return img

if QUICKDRAW_ROOT and os.path.exists(QUICKDRAW_ROOT):
    import pandas as pd
    from tqdm.notebook import tqdm

    # Find CSV files anywhere under QUICKDRAW_ROOT
    csv_files = []
    for dirpath, _, fnames in os.walk(QUICKDRAW_ROOT):
        for f in fnames:
            if f.endswith('.csv'):
                csv_files.append(os.path.join(dirpath, f))
    csv_files = sorted(csv_files)
    print(f'\nQuickDraw: {len(csv_files)} CSV files found')

    if csv_files:
        qd_count = 0
        max_per_cat = 30
        max_total   = 1000
        for csv_f in tqdm(csv_files[:40], desc='QuickDraw categories'):
            if qd_count >= max_total: break
            try:
                df = pd.read_csv(csv_f, nrows=max_per_cat + 5)
                draw_col = 'drawing' if 'drawing' in df.columns else df.columns[-1]
                rec_col  = 'recognized' if 'recognized' in df.columns else None
                if rec_col:
                    df = df[df[rec_col] == True]
                for _, row in df.head(max_per_cat).iterrows():
                    if qd_count >= max_total: break
                    pil = render_quickdraw_strokes(row[draw_col])
                    out = os.path.join(QUICKDRAW_DIR, f'qd_{qd_count:05d}.png')
                    pil.save(out)
                    quickdraw_sketch_paths.append(out)
                    qd_count += 1
            except Exception:
                continue
        print(f'QuickDraw rendered: {len(quickdraw_sketch_paths)} images')
    else:
        # Maybe it's already PNG images
        quickdraw_sketch_paths = collect_images(QUICKDRAW_ROOT)
        print(f'QuickDraw PNG images found directly: {len(quickdraw_sketch_paths)}')


# ════════════════════════════════════════════════════════════
#  COMBINE & FINAL COUNTS
# ════════════════════════════════════════════════════════════
import random
all_sketch = sketchy_sketch_paths + tuberlin_sketch_paths + quickdraw_sketch_paths
all_photo  = photo_paths_raw

random.shuffle(all_sketch)
random.shuffle(all_photo)

SUBSET_SKETCH = 3000
SUBSET_PHOTO  = 3000
sketch_paths = all_sketch[:SUBSET_SKETCH]
photo_paths  = all_photo[:SUBSET_PHOTO]

print('\n' + '='*55)
print('  FINAL DOMAIN SUMMARY')
print('='*55)
print(f'  Domain A (Sketch) : {len(sketch_paths)} images')
print(f'    Sketchy sketches : {len(sketchy_sketch_paths)}')
print(f'    TU-Berlin        : {len(tuberlin_sketch_paths)}')
print(f'    QuickDraw        : {len(quickdraw_sketch_paths)}')
print(f'  Domain B (Photo)  : {len(photo_paths)} images')
print('='*55)

if len(sketch_paths) == 0 or len(photo_paths) == 0:
    print('\nWARNING: One or both domains are empty!')
    print('The full_tree() output above shows what was actually downloaded.')
    print('Adjust the keyword search in find_folders_by_keyword() accordingly.')
else:
    print('\nReady for training!')

=== Collecting Domain B: PHOTOS ===
Photo-like dirs found: []
Total photos from Sketchy: 0

Few/no photos found in Sketchy. Downloading fallback photo dataset...
Using Colab cache for faster access to the 'animals10' dataset.
Fallback photos path: /kaggle/input/animals10
[DIR] ./
  translate.py  (0 KB)
  [DIR] raw-img/
    [DIR] raw-img/cane/
      OIF-e2bexWrojgtQnAPPcUfOWQ.jpeg  (9 KB)
      OIP---A27bIBcUgX1qkbpZOPswHaFS.jpeg  (9 KB)
      OIP---ZIdwfUcJeVxnh47zppcQHaFj.jpeg  (12 KB)
      OIP---ZRsOF7zsMqhW30WeF8-AHaFj.jpeg  (9 KB)
      OIP---_cJbI6Ei26w5bW1urHewHaCf.jpeg  (8 KB)
      ... and 4858 more files
    [DIR] raw-img/cavallo/
      OIP---MGqQIhmz3OEPYP-46_xwHaFj.jpeg  (15 KB)
      OIP---sK_NCo5VFiDavIY-pUdgHaFB.jpeg  (14 KB)
      OIP--1hNp1smpwT06GTxZbUVwgHaFi.jpeg  (15 KB)
      OIP--1iQSsCUgn4E10K_tST_QwHaHa.jpeg  (15 KB)
      OIP--2wbTZUIHhYse3zlp1S4WQHaJ3.jpeg  (12 KB)
      ... and 2618 more files
    [DIR] raw-img/elefante/
      OIP---LeldVL441fx5S66TGgVQAAAA.j

TU-Berlin:   0%|          | 0/2000 [00:00<?, ?it/s]

TU-Berlin saved: 2000

  FINAL DOMAIN SUMMARY
  Domain A (Sketch) : 2000 images
    Sketchy sketches : 0
    TU-Berlin        : 2000
    QuickDraw        : 0
  Domain B (Photo)  : 3000 images

Ready for training!


## Cell 6 — Collect Domain A (Sketches) and Domain B (Photos)

In [6]:
import gc
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def preview_domain(paths, title, n=6):
    if not paths:
        print(f'{title}: NO IMAGES TO SHOW'); return
    fig, axes = plt.subplots(1, n, figsize=(n*2.5, 2.8))
    fig.suptitle(title, fontsize=12, fontweight='bold')
    for i, ax in enumerate(axes):
        try:
            img = Image.open(paths[i]).convert('RGB').resize((128,128))
            ax.imshow(img); ax.axis('off')
        except Exception as e:
            ax.text(0.5, 0.5, str(e)[:20], ha='center', fontsize=6)
            ax.axis('off')
    plt.tight_layout()
    safe = title.replace(' ','_').replace('/','')
    plt.savefig(f'/kaggle/working/outputs/{safe}.png', dpi=100)
    plt.show(); plt.close('all'); gc.collect()

if sketch_paths: preview_domain(sketch_paths[:6], 'Domain A - Sketches (sample)')
if photo_paths:  preview_domain(photo_paths[:6],  'Domain B - Photos (sample)')

## Cell 7 — Domain Preview (Sanity Check)

In [7]:
def preview_domain(paths, title, n=6):
    fig, axes = plt.subplots(1, n, figsize=(n*2.5, 2.8))
    fig.suptitle(title, fontsize=12, fontweight='bold')
    for i, ax in enumerate(axes):
        try:
            img = Image.open(paths[i]).convert('RGB').resize((128,128))
            ax.imshow(img); ax.axis('off')
        except Exception as e:
            ax.text(0.5, 0.5, 'err', ha='center'); ax.axis('off')
    plt.tight_layout()
    safe = title.replace(' ','_')
    plt.savefig(os.path.join(OUT_DIR, f'{safe}.png'), dpi=100)
    plt.show(); plt.close('all'); gc.collect()

if sketch_paths: preview_domain(sketch_paths[:6], 'Domain A - Sketches')
if photo_paths:  preview_domain(photo_paths[:6],  'Domain B - Photos')

## Cell 8 — Unpaired Datasets & DataLoaders

In [8]:
class UnpairedDataset(Dataset):
    """
    Unpaired dataset for CycleGAN.
    Domain A and Domain B are loaded independently — no matching.
    Returns random pairs each epoch via independent random indexing.
    """
    def __init__(self, paths_A, paths_B, img_size=128, augment=True):
        self.A = paths_A
        self.B = paths_B
        self.augment = augment
        self.tf = T.Compose([
            T.Resize((img_size, img_size), T.InterpolationMode.BICUBIC),
            T.ToTensor(),
            T.Normalize([0.5]*3, [0.5]*3),   # -> [-1, 1]
        ])
        self.aug = T.Compose([
            T.Resize(int(img_size * 1.12), T.InterpolationMode.BICUBIC),
            T.RandomCrop(img_size),
            T.RandomHorizontalFlip(),
        ])

    def __len__(self):
        return max(len(self.A), len(self.B))

    def _load(self, path):
        try:
            return Image.open(path).convert('RGB')
        except Exception:
            return Image.new('RGB', (IMG_SIZE, IMG_SIZE), 128)

    def __getitem__(self, idx):
        img_A = self._load(self.A[idx % len(self.A)])
        img_B = self._load(self.B[idx % len(self.B)])
        if self.augment:
            img_A = self.aug(img_A)
            img_B = self.aug(img_B)
        return self.tf(img_A), self.tf(img_B)


# ── Train/Val split ───────────────────────────────────────────────
val_n = max(50, int(min(len(sketch_paths), len(photo_paths)) * 0.1))

tr_sketch = sketch_paths[val_n:]
tr_photo  = photo_paths[val_n:]
va_sketch = sketch_paths[:val_n]
va_photo  = photo_paths[:val_n]

tr_ds = UnpairedDataset(tr_sketch, tr_photo, IMG_SIZE, augment=True)
va_ds = UnpairedDataset(va_sketch, va_photo, IMG_SIZE, augment=False)

kw = dict(num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,
                       drop_last=True, **kw)
va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False, **kw)

print(f'Train : {len(tr_ds)} pairs  ({len(tr_loader)} batches)')
print(f'Val   : {len(va_ds)} pairs  ({len(va_loader)} batches)')

# Quick shape check
a_batch, b_batch = next(iter(tr_loader))
print(f'Batch shapes: A={a_batch.shape}  B={b_batch.shape}')
print(f'Value range : [{a_batch.min():.2f}, {a_batch.max():.2f}]')

Train : 2800 pairs  (700 batches)
Val   : 200 pairs  (50 batches)
Batch shapes: A=torch.Size([4, 3, 128, 128])  B=torch.Size([4, 3, 128, 128])
Value range : [-0.97, 1.00]


## Cell 9 — Generator: ResNet-based Architecture

In [9]:
class ResBlock(nn.Module):
    """ResNet residual block with reflection padding."""
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, 3, bias=False),
            nn.InstanceNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, 3, bias=False),
            nn.InstanceNorm2d(channels),
        )

    def forward(self, x):
        return x + self.block(x)   # residual connection


class ResNetGenerator(nn.Module):
    """
    CycleGAN Generator: ResNet-based.

    Architecture for 128x128 input:
      c7s1-64  -> d128 -> d256  -> R256 x n_blocks -> u128 -> u64 -> c7s1-3 + Tanh

    Uses InstanceNorm (not BatchNorm) — critical for CycleGAN.
    Uses ReflectionPad to avoid border artifacts.
    """
    def __init__(self, in_ch=3, out_ch=3, ngf=64, n_blocks=6):
        super().__init__()
        # Encoder
        layers = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(in_ch, ngf, 7, bias=False),      # c7s1-64
            nn.InstanceNorm2d(ngf),
            nn.ReLU(inplace=True),
            # d128
            nn.Conv2d(ngf,    ngf*2,  3, stride=2, padding=1, bias=False),
            nn.InstanceNorm2d(ngf*2),
            nn.ReLU(inplace=True),
            # d256
            nn.Conv2d(ngf*2,  ngf*4,  3, stride=2, padding=1, bias=False),
            nn.InstanceNorm2d(ngf*4),
            nn.ReLU(inplace=True),
        ]
        # ResNet blocks (transformation)
        for _ in range(n_blocks):
            layers.append(ResBlock(ngf*4))

        # Decoder
        layers += [
            # u128
            nn.ConvTranspose2d(ngf*4, ngf*2, 3, stride=2,
                               padding=1, output_padding=1, bias=False),
            nn.InstanceNorm2d(ngf*2),
            nn.ReLU(inplace=True),
            # u64
            nn.ConvTranspose2d(ngf*2, ngf,   3, stride=2,
                               padding=1, output_padding=1, bias=False),
            nn.InstanceNorm2d(ngf),
            nn.ReLU(inplace=True),
            # c7s1-3
            nn.ReflectionPad2d(3),
            nn.Conv2d(ngf, out_ch, 7),
            nn.Tanh()
        ]
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


# Shape test
_G = ResNetGenerator(n_blocks=N_RESBLOCKS).to(device)
_x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
with torch.no_grad(): _y = _G(_x)
print(f'Generator   : {tuple(_x.shape)} -> {tuple(_y.shape)}')
print(f'Params      : {sum(p.numel() for p in _G.parameters()):,}')
del _G, _x, _y

Generator   : (1, 3, 128, 128) -> (1, 3, 128, 128)
Params      : 7,833,987


## Cell 10 — Discriminator: PatchGAN

In [10]:
class PatchGANDiscriminator(nn.Module):
    """
    70x70 PatchGAN Discriminator for CycleGAN.
    Classifies each patch as real or fake (NOT conditioned on input).
    Uses InstanceNorm (no BatchNorm) for consistency with generators.
    """
    def __init__(self, in_ch=3, ndf=64):
        super().__init__()
        def conv_block(in_c, out_c, stride=2, norm=True):
            layers = [nn.Conv2d(in_c, out_c, 4, stride=stride, padding=1, bias=not norm)]
            if norm: layers.append(nn.InstanceNorm2d(out_c))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers

        self.net = nn.Sequential(
            *conv_block(in_ch, ndf,    norm=False),   # 128->64
            *conv_block(ndf,   ndf*2),                # 64->32
            *conv_block(ndf*2, ndf*4),                # 32->16
            *conv_block(ndf*4, ndf*8, stride=1),      # 16->15 (stride 1 for 70px patch)
            nn.Conv2d(ndf*8, 1, 4, padding=1),        # 15->14 (patch logit map)
        )

    def forward(self, x):
        return self.net(x)


# Shape test
_D = PatchGANDiscriminator().to(device)
_x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
with torch.no_grad(): _o = _D(_x)
print(f'Discriminator : {tuple(_x.shape)} -> patch map {tuple(_o.shape)}')
print(f'Params        : {sum(p.numel() for p in _D.parameters()):,}')
del _D, _x, _o

Discriminator : (1, 3, 128, 128) -> patch map (1, 1, 14, 14)
Params        : 2,763,841


## Cell 11 — Replay Buffer (Discriminator Stabilisation)

In [11]:
class ImageBuffer:
    """
    Replay buffer for discriminator training.
    Stores previously generated images and returns a mix of
    past and current fakes to stabilise D training.
    Classic CycleGAN technique from Shrivastava et al. 2017.
    """
    def __init__(self, max_size=50):
        self.max_size = max_size
        self.data = []

    def push_and_pop(self, data):
        result = []
        for element in data:
            if len(self.data) < self.max_size:
                self.data.append(element)
                result.append(element)
            else:
                if random.random() > 0.5:
                    idx = random.randint(0, self.max_size - 1)
                    result.append(self.data[idx].clone())
                    self.data[idx] = element
                else:
                    result.append(element)
        return torch.stack(result)


print('ImageBuffer (replay buffer) ready.')

ImageBuffer (replay buffer) ready.


## Cell 12 — CycleGAN Loss Functions

In [12]:
class CycleGANLoss:
    """
    Three loss components:

    A) Adversarial Loss (LSGAN — MSE instead of BCE)
       MSE is more stable than BCE for CycleGAN
       D_A: classifies real/fake sketches
       D_B: classifies real/fake photos

    B) Cycle Consistency Loss
       Forward : A -> G_AB(A) -> G_BA(G_AB(A)) ≈ A
       Backward: B -> G_BA(B) -> G_AB(G_BA(B)) ≈ B
       Weight : lambda_cycle = 10

    C) Identity Loss
       G_AB(B) ≈ B  (preserve colour/style when input is already target domain)
       G_BA(A) ≈ A
       Weight : lambda_id = 5 (half of cycle loss)
    """
    def __init__(self, lambda_cycle=10.0, lambda_id=5.0):
        self.lambda_cycle = lambda_cycle
        self.lambda_id    = lambda_id
        self.mse = nn.MSELoss()    # LSGAN objective
        self.l1  = nn.L1Loss()     # cycle + identity

    def adv_real(self, pred):
        """Discriminator: real images should output 1."""
        return self.mse(pred, torch.ones_like(pred))

    def adv_fake(self, pred):
        """Discriminator: fake images should output 0."""
        return self.mse(pred, torch.zeros_like(pred))

    def adv_gen(self, pred):
        """Generator: fool D -> predict 1 for fakes."""
        return self.mse(pred, torch.ones_like(pred))

    def cycle(self, real, reconstructed):
        return self.l1(reconstructed, real) * self.lambda_cycle

    def identity(self, real, identity_out):
        return self.l1(identity_out, real) * self.lambda_id


print('CycleGAN loss functions ready.')

CycleGAN loss functions ready.


## Cell 13 — Model Builder & Weight Init

In [13]:
def weights_init(m):
    cn = m.__class__.__name__
    if 'Conv' in cn:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if m.bias is not None:
            nn.init.constant_(m.bias.data, 0.0)
    elif 'InstanceNorm' in cn and m.weight is not None:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0.0)


def build_cyclegan():
    # Generators
    G_AB = ResNetGenerator(n_blocks=N_RESBLOCKS).to(device)  # Sketch -> Photo
    G_BA = ResNetGenerator(n_blocks=N_RESBLOCKS).to(device)  # Photo  -> Sketch
    # Discriminators
    D_A  = PatchGANDiscriminator().to(device)   # classifies real/fake sketches
    D_B  = PatchGANDiscriminator().to(device)   # classifies real/fake photos

    for net in [G_AB, G_BA, D_A, D_B]:
        net.apply(weights_init)

    if N_GPUS > 1:
        G_AB = nn.DataParallel(G_AB)
        G_BA = nn.DataParallel(G_BA)
        D_A  = nn.DataParallel(D_A)
        D_B  = nn.DataParallel(D_B)
        print(f'DataParallel on {N_GPUS} GPUs')

    # Optimizers — generators share one optimizer (standard CycleGAN)
    opt_G = optim.Adam(
        itertools.chain(G_AB.parameters(), G_BA.parameters()),
        lr=LR, betas=BETAS
    )
    opt_D_A = optim.Adam(D_A.parameters(), lr=LR, betas=BETAS)
    opt_D_B = optim.Adam(D_B.parameters(), lr=LR, betas=BETAS)

    # Linear LR decay: keep LR constant for DECAY_START epochs,
    # then linearly decay to 0 over remaining epochs
    def lr_lambda(epoch):
        if epoch < DECAY_START:
            return 1.0
        return max(0.0, 1.0 - (epoch - DECAY_START) / (NUM_EPOCHS - DECAY_START + 1))

    sch_G   = optim.lr_scheduler.LambdaLR(opt_G,   lr_lambda)
    sch_D_A = optim.lr_scheduler.LambdaLR(opt_D_A, lr_lambda)
    sch_D_B = optim.lr_scheduler.LambdaLR(opt_D_B, lr_lambda)

    criterion = CycleGANLoss(LAMBDA_CYCLE, LAMBDA_ID)
    scaler    = GradScaler('cuda')

    buf_A = ImageBuffer(BUFFER_SIZE)  # buffer for fake sketches
    buf_B = ImageBuffer(BUFFER_SIZE)  # buffer for fake photos

    return (G_AB, G_BA, D_A, D_B,
            opt_G, opt_D_A, opt_D_B,
            sch_G, sch_D_A, sch_D_B,
            criterion, scaler, buf_A, buf_B)


print('Model builder ready.')

Model builder ready.


## Cell 14 — Training Step

In [14]:
def train_step(real_A, real_B,
               G_AB, G_BA, D_A, D_B,
               opt_G, opt_D_A, opt_D_B,
               criterion, scaler, buf_A, buf_B):
    """
    One batch training step for CycleGAN.

    Forward pass:
      fake_B  = G_AB(real_A)          Sketch -> Photo
      fake_A  = G_BA(real_B)          Photo  -> Sketch
      rec_A   = G_BA(fake_B)          Photo  -> Sketch (cycle)
      rec_B   = G_AB(fake_A)          Sketch -> Photo  (cycle)
      id_A    = G_BA(real_A)          Identity for sketch domain
      id_B    = G_AB(real_B)          Identity for photo domain
    """
    real_A = real_A.to(device, non_blocking=True)
    real_B = real_B.to(device, non_blocking=True)

    # ── STEP 1: Update Generators (G_AB + G_BA together) ──────────
    opt_G.zero_grad(set_to_none=True)

    with autocast(device_type='cuda'):
        # Generate fakes
        fake_B = G_AB(real_A)    # Sketch -> Photo
        fake_A = G_BA(real_B)    # Photo  -> Sketch

        # Cycle reconstructions
        rec_A  = G_BA(fake_B)    # -> Sketch (should match real_A)
        rec_B  = G_AB(fake_A)    # -> Photo  (should match real_B)

        # Identity outputs
        id_A   = G_BA(real_A)    # G_BA on sketch -> should be same sketch
        id_B   = G_AB(real_B)    # G_AB on photo  -> should be same photo

        # Adversarial: G wants D to output 1 for fakes
        loss_G_AB = criterion.adv_gen(D_B(fake_B))
        loss_G_BA = criterion.adv_gen(D_A(fake_A))

        # Cycle consistency
        loss_cyc_A = criterion.cycle(real_A, rec_A)
        loss_cyc_B = criterion.cycle(real_B, rec_B)

        # Identity
        loss_id_A  = criterion.identity(real_A, id_A)
        loss_id_B  = criterion.identity(real_B, id_B)

        loss_G = (loss_G_AB + loss_G_BA
                  + loss_cyc_A + loss_cyc_B
                  + loss_id_A  + loss_id_B)

    scaler.scale(loss_G).backward()
    scaler.step(opt_G)

    # ── STEP 2: Update D_A (sketch discriminator) ──────────────────
    opt_D_A.zero_grad(set_to_none=True)
    with autocast(device_type='cuda'):
        # Use buffered fakes for stable D training
        fake_A_buf = buf_A.push_and_pop(fake_A.detach())
        loss_D_A = 0.5 * (criterion.adv_real(D_A(real_A)) +
                          criterion.adv_fake(D_A(fake_A_buf)))
    scaler.scale(loss_D_A).backward()
    scaler.step(opt_D_A)

    # ── STEP 3: Update D_B (photo discriminator) ───────────────────
    opt_D_B.zero_grad(set_to_none=True)
    with autocast(device_type='cuda'):
        fake_B_buf = buf_B.push_and_pop(fake_B.detach())
        loss_D_B = 0.5 * (criterion.adv_real(D_B(real_B)) +
                          criterion.adv_fake(D_B(fake_B_buf)))
    scaler.scale(loss_D_B).backward()
    scaler.step(opt_D_B)

    scaler.update()

    return {
        'G'    : loss_G.item(),
        'D_A'  : loss_D_A.item(),
        'D_B'  : loss_D_B.item(),
        'cyc_A': loss_cyc_A.item(),
        'cyc_B': loss_cyc_B.item(),
        'id_A' : loss_id_A.item(),
        'id_B' : loss_id_B.item(),
        'adv_G': (loss_G_AB + loss_G_BA).item(),
    }


def train_epoch(loader, G_AB, G_BA, D_A, D_B,
                opt_G, opt_D_A, opt_D_B,
                criterion, scaler, buf_A, buf_B):
    G_AB.train(); G_BA.train(); D_A.train(); D_B.train()
    totals = {k: 0.0 for k in ['G','D_A','D_B','cyc_A','cyc_B','id_A','id_B','adv_G']}

    for real_A, real_B in loader:
        losses = train_step(real_A, real_B,
                            G_AB, G_BA, D_A, D_B,
                            opt_G, opt_D_A, opt_D_B,
                            criterion, scaler, buf_A, buf_B)
        for k in totals:
            totals[k] += losses[k]

    n = len(loader)
    return {k: v/n for k, v in totals.items()}


print('Training step ready.')

Training step ready.


## Cell 15 — Evaluation (SSIM & PSNR)

In [15]:
@torch.no_grad()
def evaluate(G_AB, G_BA, loader):
    """
    Compute SSIM and PSNR for:
    - Sketch->Photo->Sketch (cycle A)
    - Photo->Sketch->Photo  (cycle B)
    """
    G_AB.eval(); G_BA.eval()
    ssim_cyc_A, psnr_cyc_A = [], []
    ssim_cyc_B, psnr_cyc_B = [], []

    for real_A, real_B in loader:
        real_A = real_A.to(device)
        real_B = real_B.to(device)

        with autocast(device_type='cuda'):
            fake_B = G_AB(real_A)        # Sketch -> Photo
            rec_A  = G_BA(fake_B)        # -> Sketch (reconstruct)
            fake_A = G_BA(real_B)        # Photo -> Sketch
            rec_B  = G_AB(fake_A)        # -> Photo (reconstruct)

        def to_np(t):
            return (t.cpu().float().numpy() * 0.5 + 0.5).clip(0, 1)

        for r, c in zip(to_np(real_A), to_np(rec_A)):
            r = r.transpose(1,2,0); c = c.transpose(1,2,0)
            ssim_cyc_A.append(compute_ssim(r, c, channel_axis=2, data_range=1.0))
            psnr_cyc_A.append(compute_psnr(r, c, data_range=1.0))

        for r, c in zip(to_np(real_B), to_np(rec_B)):
            r = r.transpose(1,2,0); c = c.transpose(1,2,0)
            ssim_cyc_B.append(compute_ssim(r, c, channel_axis=2, data_range=1.0))
            psnr_cyc_B.append(compute_psnr(r, c, data_range=1.0))

    return {
        'ssim_cyc_A': float(np.mean(ssim_cyc_A)),
        'psnr_cyc_A': float(np.mean(psnr_cyc_A)),
        'ssim_cyc_B': float(np.mean(ssim_cyc_B)),
        'psnr_cyc_B': float(np.mean(psnr_cyc_B)),
    }


def save_ckpt(G_AB, G_BA, D_A, D_B,
              opt_G, opt_D_A, opt_D_B, epoch, metrics, path):
    torch.save({
        'epoch'  : epoch,
        'metrics': metrics,
        'G_AB'   : unwrap(G_AB).state_dict(),
        'G_BA'   : unwrap(G_BA).state_dict(),
        'D_A'    : unwrap(D_A).state_dict(),
        'D_B'    : unwrap(D_B).state_dict(),
        'opt_G'  : opt_G.state_dict(),
        'opt_D_A': opt_D_A.state_dict(),
        'opt_D_B': opt_D_B.state_dict(),
    }, path)


print('Evaluation & checkpoint utilities ready.')

Evaluation & checkpoint utilities ready.


## Cell 16 — Visualisation Utility

In [16]:
@torch.no_grad()
def visualize_cyclegan(G_AB, G_BA, loader, title, n=6, save_path=None):
    """
    4 rows x n columns:
      Row 1: Real Sketch (Domain A input)
      Row 2: Translated Photo G_AB(A)  [Sketch -> Photo]
      Row 3: Real Photo (Domain B input)
      Row 4: Translated Sketch G_BA(B) [Photo -> Sketch]
    """
    G_AB.eval(); G_BA.eval()

    real_A_list, fake_B_list = [], []
    real_B_list, fake_A_list = [], []
    rec_A_list,  rec_B_list  = [], []

    for real_A, real_B in loader:
        real_A = real_A.to(device)
        real_B = real_B.to(device)
        with autocast(device_type='cuda'):
            fake_B = G_AB(real_A)
            fake_A = G_BA(real_B)
            rec_A  = G_BA(fake_B)
            rec_B  = G_AB(fake_A)
        real_A_list.append(real_A.cpu().float())
        fake_B_list.append(fake_B.cpu().float())
        real_B_list.append(real_B.cpu().float())
        fake_A_list.append(fake_A.cpu().float())
        rec_A_list.append(rec_A.cpu().float())
        rec_B_list.append(rec_B.cpu().float())
        if sum(x.size(0) for x in real_A_list) >= n:
            break

    def cat(lst): return torch.cat(lst)[:n]
    def denorm(t): return (t * 0.5 + 0.5).clamp(0,1).permute(1,2,0).numpy()

    rows = [
        ('Real Sketch (A)',        cat(real_A_list)),
        ('Translated Photo (B)',   cat(fake_B_list)),
        ('Reconstructed Sketch',   cat(rec_A_list)),
        ('Real Photo (B)',         cat(real_B_list)),
        ('Translated Sketch (A)',  cat(fake_A_list)),
        ('Reconstructed Photo',    cat(rec_B_list)),
    ]

    fig, axes = plt.subplots(6, n, figsize=(n*2.5, 14))
    fig.suptitle(title, fontsize=13, fontweight='bold', y=1.01)

    for r, (lbl, imgs) in enumerate(rows):
        axes[r, 0].set_ylabel(lbl, fontsize=8, fontweight='bold',
                               rotation=0, labelpad=75, va='center')
        for c in range(min(n, imgs.size(0))):
            axes[r, c].imshow(np.clip(denorm(imgs[c]), 0, 1))
            axes[r, c].axis('off')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=130, bbox_inches='tight')
        print(f'Saved -> {save_path}')
    plt.show(); plt.close('all'); gc.collect()


print('Visualisation utility ready.')

Visualisation utility ready.


## Cell 17 — Master Training Loop

In [17]:
# Build all models
(G_AB, G_BA, D_A, D_B,
 opt_G, opt_D_A, opt_D_B,
 sch_G, sch_D_A, sch_D_B,
 criterion, scaler, buf_A, buf_B) = build_cyclegan()

# History
hist = {
    'G':[], 'D_A':[], 'D_B':[], 'adv_G':[],
    'cyc_A':[], 'cyc_B':[], 'id_A':[], 'id_B':[],
    'ssim_cyc_A':[], 'psnr_cyc_A':[],
    'ssim_cyc_B':[], 'psnr_cyc_B':[],
    'epochs':[]
}

best_ssim  = -1.0
EVAL_EVERY = 5
VIS_EVERY  = 10

print(f'{'='*65}')
print(f'  CycleGAN Training  (max {NUM_EPOCHS} epochs)')
print(f'  LAMBDA_CYCLE={LAMBDA_CYCLE}  LAMBDA_ID={LAMBDA_ID}')
print(f'{'='*65}')

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    losses = train_epoch(
        tr_loader, G_AB, G_BA, D_A, D_B,
        opt_G, opt_D_A, opt_D_B,
        criterion, scaler, buf_A, buf_B
    )

    sch_G.step(); sch_D_A.step(); sch_D_B.step()

    # Evaluate periodically
    if epoch % EVAL_EVERY == 0 or epoch == NUM_EPOCHS:
        metrics = evaluate(G_AB, G_BA, va_loader)
        avg_ssim = (metrics['ssim_cyc_A'] + metrics['ssim_cyc_B']) / 2

        if avg_ssim > best_ssim:
            best_ssim = avg_ssim
            save_ckpt(G_AB, G_BA, D_A, D_B,
                      opt_G, opt_D_A, opt_D_B, epoch, metrics,
                      os.path.join(CKPT_DIR, 'best.pt'))
            print(f'  NEW BEST SSIM(avg)={best_ssim:.4f}  (checkpoint saved)')
    else:
        metrics = {k: hist[k][-1] if hist[k] else 0.0
                   for k in ['ssim_cyc_A','psnr_cyc_A','ssim_cyc_B','psnr_cyc_B']}

    # Store history
    for k in ['G','D_A','D_B','cyc_A','cyc_B','id_A','id_B','adv_G']:
        hist[k].append(losses[k])
    for k in ['ssim_cyc_A','psnr_cyc_A','ssim_cyc_B','psnr_cyc_B']:
        hist[k].append(metrics[k])
    hist['epochs'].append(epoch)

    elapsed = time.time() - t0
    print(f'Ep {epoch:3d}/{NUM_EPOCHS} '
          f'| G={losses["G"]:.4f} D_A={losses["D_A"]:.4f} D_B={losses["D_B"]:.4f} '
          f'| CycA={losses["cyc_A"]:.4f} CycB={losses["cyc_B"]:.4f} '
          f'| SSIM_A={metrics["ssim_cyc_A"]:.3f} SSIM_B={metrics["ssim_cyc_B"]:.3f} '
          f'| {elapsed:.1f}s')

    # Periodic checkpoint
    if epoch % SAVE_EVERY == 0:
        cp = os.path.join(CKPT_DIR, f'ep{epoch:04d}.pt')
        save_ckpt(G_AB, G_BA, D_A, D_B,
                  opt_G, opt_D_A, opt_D_B, epoch, metrics, cp)
        print(f'  [ckpt] {cp}')

    # Visualise periodically
    if epoch % VIS_EVERY == 0 or epoch == NUM_EPOCHS:
        visualize_cyclegan(
            G_AB, G_BA, va_loader,
            title=f'CycleGAN — Epoch {epoch}',
            n=6,
            save_path=os.path.join(OUT_DIR, f'vis_ep{epoch:04d}.png')
        )

print(f'{'='*65}')
print(f'Training complete. Best SSIM(avg) = {best_ssim:.4f}')
print(f'{'='*65}')

  CycleGAN Training  (max 20 epochs)
  LAMBDA_CYCLE=10.0  LAMBDA_ID=5.0
Ep   1/20 | G=6.3607 D_A=0.2035 D_B=0.3321 | CycA=0.5818 CycB=3.0551 | SSIM_A=0.000 SSIM_B=0.000 | 137.6s
Ep   2/20 | G=5.3074 D_A=0.1010 D_B=0.1760 | CycA=0.3882 CycB=2.4974 | SSIM_A=0.000 SSIM_B=0.000 | 130.2s
Ep   3/20 | G=5.0530 D_A=0.0815 D_B=0.1546 | CycA=0.3698 CycB=2.3137 | SSIM_A=0.000 SSIM_B=0.000 | 130.3s
Ep   4/20 | G=4.9584 D_A=0.0564 D_B=0.1491 | CycA=0.3450 CycB=2.2076 | SSIM_A=0.000 SSIM_B=0.000 | 130.3s
  NEW BEST SSIM(avg)=0.7733  (checkpoint saved)
Ep   5/20 | G=4.7459 D_A=0.0686 D_B=0.1472 | CycA=0.3361 CycB=2.1125 | SSIM_A=0.940 SSIM_B=0.606 | 136.8s
Ep   6/20 | G=4.7284 D_A=0.0448 D_B=0.1380 | CycA=0.3233 CycB=2.0392 | SSIM_A=0.940 SSIM_B=0.606 | 130.2s
Ep   7/20 | G=4.6663 D_A=0.0368 D_B=0.1397 | CycA=0.3054 CycB=1.9993 | SSIM_A=0.940 SSIM_B=0.606 | 131.1s
Ep   8/20 | G=4.6046 D_A=0.0172 D_B=0.1397 | CycA=0.2930 CycB=1.9153 | SSIM_A=0.940 SSIM_B=0.606 | 130.5s
Ep   9/20 | G=4.4674 D_A=0.0338 

## Cell 18 — Training Loss Plots

In [18]:
eps = hist['epochs']
fig, axes = plt.subplots(3, 3, figsize=(18, 12))
fig.suptitle('CycleGAN Training Logs', fontsize=15, fontweight='bold')

def _p(ax, y, lbl, col):
    ax.plot(eps, y, color=col, lw=2)
    ax.set_title(lbl, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.grid(alpha=0.3)

_p(axes[0,0], hist['G'],       'Generator Loss (total)',    '#E74C3C')
_p(axes[0,1], hist['D_A'],     'D_A Loss (Sketch)',         '#3498DB')
_p(axes[0,2], hist['D_B'],     'D_B Loss (Photo)',          '#2980B9')
_p(axes[1,0], hist['adv_G'],   'G Adversarial Loss',        '#9B59B6')
_p(axes[1,1], hist['cyc_A'],   'Cycle Loss A (Sketch→Photo→Sketch)', '#27AE60')
_p(axes[1,2], hist['cyc_B'],   'Cycle Loss B (Photo→Sketch→Photo)',  '#16A085')
_p(axes[2,0], hist['id_A'],    'Identity Loss A',           '#F39C12')
_p(axes[2,1], hist['ssim_cyc_A'], 'SSIM Cycle A (↑ better)', '#8E44AD')
_p(axes[2,2], hist['ssim_cyc_B'], 'SSIM Cycle B (↑ better)', '#D35400')

plt.tight_layout()
path = os.path.join(OUT_DIR, 'training_logs.png')
plt.savefig(path, dpi=120, bbox_inches='tight')
plt.show(); plt.close('all'); gc.collect()
print(f'Saved -> {path}')

Saved -> /kaggle/working/outputs/training_logs.png


## Cell 19 — Final Visual Results (5+ qualitative examples)

In [19]:
visualize_cyclegan(
    G_AB, G_BA, va_loader,
    title='CycleGAN Final Results — Sketch ↔ Photo Translation',
    n=8,
    save_path=os.path.join(OUT_DIR, 'final_results.png')
)

Saved -> /kaggle/working/outputs/final_results.png


## Cell 20 — Quantitative Evaluation (SSIM & PSNR)

In [20]:
@torch.no_grad()
def full_quantitative_eval(G_AB, G_BA, loader):
    G_AB.eval(); G_BA.eval()
    ssim_A, psnr_A = [], []
    ssim_B, psnr_B = [], []
    ssim_trA, psnr_trA = [], []   # translated A (sketch->photo quality vs real photo)

    for real_A, real_B in tqdm(loader, desc='Evaluating'):
        real_A = real_A.to(device)
        real_B = real_B.to(device)
        with autocast(device_type='cuda'):
            fake_B = G_AB(real_A)
            rec_A  = G_BA(fake_B)
            fake_A = G_BA(real_B)
            rec_B  = G_AB(fake_A)

        def np_(t): return (t.cpu().float().numpy()*0.5+0.5).clip(0,1)

        for r, c in zip(np_(real_A), np_(rec_A)):
            r=r.transpose(1,2,0); c=c.transpose(1,2,0)
            ssim_A.append(compute_ssim(r,c,channel_axis=2,data_range=1.0))
            psnr_A.append(compute_psnr(r,c,data_range=1.0))

        for r, c in zip(np_(real_B), np_(rec_B)):
            r=r.transpose(1,2,0); c=c.transpose(1,2,0)
            ssim_B.append(compute_ssim(r,c,channel_axis=2,data_range=1.0))
            psnr_B.append(compute_psnr(r,c,data_range=1.0))

    ssim_A_arr = np.array(ssim_A)
    psnr_A_arr = np.array(psnr_A)
    ssim_B_arr = np.array(ssim_B)
    psnr_B_arr = np.array(psnr_B)

    print('\n' + '='*60)
    print('  QUANTITATIVE EVALUATION: CycleGAN')
    print('='*60)
    print('  Cycle A: Sketch -> Photo -> Sketch (reconstruction)')
    print(f'    SSIM : {ssim_A_arr.mean():.4f} +/- {ssim_A_arr.std():.4f}')
    print(f'    PSNR : {psnr_A_arr.mean():.2f}dB +/- {psnr_A_arr.std():.2f}')
    print('  Cycle B: Photo -> Sketch -> Photo (reconstruction)')
    print(f'    SSIM : {ssim_B_arr.mean():.4f} +/- {ssim_B_arr.std():.4f}')
    print(f'    PSNR : {psnr_B_arr.mean():.2f}dB +/- {psnr_B_arr.std():.2f}')
    print('='*60)
    print('  Note: CycleGAN metrics measure CYCLE CONSISTENCY (reconstruction')
    print('        fidelity), not translation quality (which is unpaired).')
    print('  Higher SSIM/PSNR = better structural preservation in cycle.')
    print('='*60)

    return ssim_A_arr, psnr_A_arr, ssim_B_arr, psnr_B_arr


ssim_A_arr, psnr_A_arr, ssim_B_arr, psnr_B_arr = full_quantitative_eval(
    G_AB, G_BA, va_loader)

Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]


  QUANTITATIVE EVALUATION: CycleGAN
  Cycle A: Sketch -> Photo -> Sketch (reconstruction)
    SSIM : 0.9404 +/- 0.0212
    PSNR : 27.94dB +/- 2.05
  Cycle B: Photo -> Sketch -> Photo (reconstruction)
    SSIM : 0.7413 +/- 0.0663
    PSNR : 20.52dB +/- 1.99
  Note: CycleGAN metrics measure CYCLE CONSISTENCY (reconstruction
        fidelity), not translation quality (which is unpaired).
  Higher SSIM/PSNR = better structural preservation in cycle.


## Cell 21 — Save Weights + ZIP for Download

In [21]:
import json as json_lib

# Final unwrapped generator weights
G_AB_PATH = os.path.join(CKPT_DIR, 'G_AB_final.pt')   # Sketch -> Photo
G_BA_PATH = os.path.join(CKPT_DIR, 'G_BA_final.pt')   # Photo  -> Sketch

torch.save(unwrap(G_AB).state_dict(), G_AB_PATH)
torch.save(unwrap(G_BA).state_dict(), G_BA_PATH)
print(f'Saved: {G_AB_PATH}')
print(f'Saved: {G_BA_PATH}')

# Config JSON
config = dict(
    IMG_SIZE      = IMG_SIZE,
    N_RESBLOCKS   = N_RESBLOCKS,
    LAMBDA_CYCLE  = LAMBDA_CYCLE,
    LAMBDA_ID     = LAMBDA_ID,
    ssim_cycle_A  = round(float(ssim_A_arr.mean()), 4),
    psnr_cycle_A  = round(float(psnr_A_arr.mean()), 2),
    ssim_cycle_B  = round(float(ssim_B_arr.mean()), 4),
    psnr_cycle_B  = round(float(psnr_B_arr.mean()), 2),
)
CONFIG_PATH = '/kaggle/working/cyclegan_config.json'
with open(CONFIG_PATH, 'w') as f:
    json_lib.dump(config, f, indent=2)
print(f'Config saved: {CONFIG_PATH}')

# Bundle into ZIP
ZIP = '/kaggle/working/cyclegan_weights.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in [G_AB_PATH, G_BA_PATH, CONFIG_PATH]:
        zf.write(p, os.path.basename(p))
    for png in glob.glob(os.path.join(OUT_DIR, '*.png')):
        zf.write(png, f'outputs/{os.path.basename(png)}')

print(f'\nZIP: {ZIP}  ({os.path.getsize(ZIP)/1e6:.1f} MB)')
print('Download from Kaggle OUTPUT tab.')

Saved: /kaggle/working/ckpt_cyclegan/G_AB_final.pt
Saved: /kaggle/working/ckpt_cyclegan/G_BA_final.pt
Config saved: /kaggle/working/cyclegan_config.json

ZIP: /kaggle/working/cyclegan_weights.zip  (68.7 MB)
Download from Kaggle OUTPUT tab.


## Cell 22 — Generate HuggingFace Deployment Files

In [22]:
APP_PY = '''
# ═══════════════════════════════════════════════════════════
#  HuggingFace Gradio Space — CycleGAN Sketch <-> Photo
#  Files needed:
#    app.py  requirements.txt
#    G_AB_final.pt   (Sketch -> Photo)
#    G_BA_final.pt   (Photo  -> Sketch)
#    cyclegan_config.json
# ═══════════════════════════════════════════════════════════
import json, torch, gradio as gr, numpy as np
from PIL import Image
import torch.nn as nn
import torchvision.transforms as T

with open("cyclegan_config.json") as f:
    cfg = json.load(f)
IMG_SIZE    = cfg["IMG_SIZE"]
N_RESBLOCKS = cfg["N_RESBLOCKS"]
device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1), nn.Conv2d(ch,ch,3,bias=False),
            nn.InstanceNorm2d(ch), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.ReflectionPad2d(1), nn.Conv2d(ch,ch,3,bias=False),
            nn.InstanceNorm2d(ch))
    def forward(self, x): return x + self.block(x)

class ResNetGenerator(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, ngf=64, n_blocks=6):
        super().__init__()
        layers = [
            nn.ReflectionPad2d(3), nn.Conv2d(in_ch,ngf,7,bias=False),
            nn.InstanceNorm2d(ngf), nn.ReLU(inplace=True),
            nn.Conv2d(ngf,ngf*2,3,stride=2,padding=1,bias=False),
            nn.InstanceNorm2d(ngf*2), nn.ReLU(inplace=True),
            nn.Conv2d(ngf*2,ngf*4,3,stride=2,padding=1,bias=False),
            nn.InstanceNorm2d(ngf*4), nn.ReLU(inplace=True),
        ] + [ResBlock(ngf*4) for _ in range(n_blocks)] + [
            nn.ConvTranspose2d(ngf*4,ngf*2,3,stride=2,padding=1,output_padding=1,bias=False),
            nn.InstanceNorm2d(ngf*2), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(ngf*2,ngf,3,stride=2,padding=1,output_padding=1,bias=False),
            nn.InstanceNorm2d(ngf), nn.ReLU(inplace=True),
            nn.ReflectionPad2d(3), nn.Conv2d(ngf,out_ch,7), nn.Tanh()
        ]
        self.model = nn.Sequential(*layers)
    def forward(self, x): return self.model(x)

def load_gen(path):
    g = ResNetGenerator(n_blocks=N_RESBLOCKS).to(device)
    g.load_state_dict(torch.load(path, map_location=device))
    return g.eval()

G_AB = load_gen("G_AB_final.pt")   # Sketch -> Photo
G_BA = load_gen("G_BA_final.pt")   # Photo  -> Sketch

tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE), T.InterpolationMode.BICUBIC),
    T.ToTensor(), T.Normalize([0.5]*3, [0.5]*3)
])

def proc(pil): return tf(pil.convert("RGB")).unsqueeze(0).to(device)
def deproc(t):
    a = (t.squeeze(0).cpu().float().numpy()*0.5+0.5).clip(0,1)
    return Image.fromarray((a.transpose(1,2,0)*255).astype("uint8"))

@torch.no_grad()
def translate(img_pil, direction):
    x = proc(img_pil)
    G = G_AB if direction == "Sketch -> Photo" else G_BA
    return deproc(G(x))

@torch.no_grad()
def full_cycle(img_pil, direction):
    x = proc(img_pil)
    if direction == "Sketch -> Photo":
        translated = G_AB(x)
        reconstructed = G_BA(translated)
    else:
        translated = G_BA(x)
        reconstructed = G_AB(translated)
    return deproc(translated), deproc(reconstructed)

with gr.Blocks(title="CycleGAN Sketch <-> Photo", theme=gr.themes.Soft()) as demo:
    gr.Markdown(f"""
    # CycleGAN: Sketch ↔ Photo Domain Translation
    Unpaired image translation — no matched pairs required during training.

    | Direction | Cycle SSIM | Cycle PSNR |
    |-----------|-----------|----------|
    | Sketch→Photo→Sketch | {cfg["ssim_cycle_A"]} | {cfg["psnr_cycle_A"]}dB |
    | Photo→Sketch→Photo | {cfg["ssim_cycle_B"]} | {cfg["psnr_cycle_B"]}dB |
    """)

    with gr.Tab("Translate"):
        with gr.Row():
            with gr.Column():
                inp = gr.Image(type="pil", label="Input Image")
                dir_radio = gr.Radio(
                    ["Sketch -> Photo", "Photo -> Sketch"],
                    value="Sketch -> Photo", label="Translation Direction")
                btn = gr.Button("Translate", variant="primary")
            out = gr.Image(type="pil", label="Translated Output")
        btn.click(translate, [inp, dir_radio], out)

    with gr.Tab("Full Cycle"):
        gr.Markdown("Shows: Input -> Translated -> Reconstructed (cycle)")
        with gr.Row():
            inp2 = gr.Image(type="pil", label="Input")
            dir2 = gr.Radio(
                ["Sketch -> Photo", "Photo -> Sketch"],
                value="Sketch -> Photo", label="Direction")
        btn2 = gr.Button("Run Full Cycle", variant="primary")
        with gr.Row():
            t_out = gr.Image(type="pil", label="Translated")
            r_out = gr.Image(type="pil", label="Reconstructed")
        btn2.click(full_cycle, [inp2, dir2], [t_out, r_out])

    with gr.Tab("Architecture"):
        gr.Markdown(f"""
        ## CycleGAN Architecture
        | Component | Details |
        |-----------|--------|
        | G_AB | ResNet Generator ({N_RESBLOCKS} blocks), Sketch→Photo |
        | G_BA | ResNet Generator ({N_RESBLOCKS} blocks), Photo→Sketch |
        | D_A | PatchGAN, classifies sketch domain |
        | D_B | PatchGAN, classifies photo domain |
        | Image Size | {IMG_SIZE}×{IMG_SIZE} |

        ## Loss Functions
        | Loss | Weight | Purpose |
        |------|--------|---------|
        | Adversarial (LSGAN) | 1 | Domain realism |
        | Cycle Consistency | {cfg["LAMBDA_CYCLE"]} | Structural preservation |
        | Identity | {cfg["LAMBDA_ID"]} | Style preservation |
        """)

demo.launch()
'''

REQUIREMENTS = 'torch\ntorchvision\ngradio\nnumpy\nPillow\n'

with open(os.path.join(HF_DIR, 'app.py'), 'w') as f:           f.write(APP_PY)
with open(os.path.join(HF_DIR, 'requirements.txt'), 'w') as f: f.write(REQUIREMENTS)

for fname, src in [('G_AB_final.pt', G_AB_PATH),
                   ('G_BA_final.pt', G_BA_PATH),
                   ('cyclegan_config.json', CONFIG_PATH)]:
    shutil.copy(src, os.path.join(HF_DIR, fname))
    print(f'  Copied {fname}')

HF_ZIP = '/kaggle/working/cyclegan_hf_deploy.zip'
with zipfile.ZipFile(HF_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fpath in glob.glob(os.path.join(HF_DIR, '*')):
        zf.write(fpath, os.path.basename(fpath))

print(f'\nHF ZIP: {HF_ZIP}  ({os.path.getsize(HF_ZIP)/1e6:.1f} MB)')
print()
print('HUGGINGFACE DEPLOYMENT — 3 steps:')
print('  1. Download cyclegan_hf_deploy.zip from Kaggle OUTPUT tab')
print('  2. huggingface.co/new-space -> SDK=Gradio, Hardware=CPU Basic')
print('  3. Upload: app.py  requirements.txt')
print('     G_AB_final.pt  G_BA_final.pt  cyclegan_config.json')
print('  Space auto-builds in ~1 min!')

  Copied G_AB_final.pt
  Copied G_BA_final.pt
  Copied cyclegan_config.json

HF ZIP: /kaggle/working/cyclegan_hf_deploy.zip  (58.2 MB)

HUGGINGFACE DEPLOYMENT — 3 steps:
  1. Download cyclegan_hf_deploy.zip from Kaggle OUTPUT tab
  2. huggingface.co/new-space -> SDK=Gradio, Hardware=CPU Basic
  3. Upload: app.py  requirements.txt
     G_AB_final.pt  G_BA_final.pt  cyclegan_config.json
  Space auto-builds in ~1 min!


## Cell 23 — Live Gradio Demo (in Kaggle)

In [23]:
import gradio as gr

G_AB.eval(); G_BA.eval()

tf_live = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE), T.InterpolationMode.BICUBIC),
    T.ToTensor(), T.Normalize([0.5]*3, [0.5]*3)
])

def preproc(pil):
    return tf_live(pil.convert('RGB')).unsqueeze(0).to(device)

def deproc(t):
    a = (t.squeeze(0).cpu().float().numpy() * 0.5 + 0.5).clip(0,1)
    return Image.fromarray((a.transpose(1,2,0)*255).astype(np.uint8))

@torch.no_grad()
def translate_live(img_pil, direction):
    if img_pil is None: return None
    x = preproc(img_pil)
    G = G_AB if 'Sketch' in direction else G_BA
    return deproc(G(x))

@torch.no_grad()
def cycle_live(img_pil, direction):
    if img_pil is None: return None, None
    x = preproc(img_pil)
    if 'Sketch' in direction:
        t = G_AB(x); r = G_BA(t)
    else:
        t = G_BA(x); r = G_AB(t)
    return deproc(t), deproc(r)


with gr.Blocks(title='CycleGAN Sketch<->Photo', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# CycleGAN: Sketch ↔ Photo Domain Translation')

    with gr.Tab('Translate'):
        with gr.Row():
            with gr.Column():
                inp  = gr.Image(type='pil', label='Input Image')
                dire = gr.Radio(
                    ['Sketch -> Photo', 'Photo -> Sketch'],
                    value='Sketch -> Photo', label='Direction')
                out  = gr.Image(type='pil', label='Output')
            gr.Button('Translate', variant='primary').click(
                translate_live, [inp, dire], out)

    with gr.Tab('Full Cycle Visualisation'):
        inp2 = gr.Image(type='pil', label='Input')
        dir2 = gr.Radio(['Sketch -> Photo','Photo -> Sketch'],
                        value='Sketch -> Photo', label='Direction')
        with gr.Row():
            t2 = gr.Image(type='pil', label='Translated')
            r2 = gr.Image(type='pil', label='Reconstructed (cycle)')
        gr.Button('Run Cycle', variant='primary').click(
            cycle_live, [inp2, dir2], [t2, r2])

    with gr.Tab('Metrics'):
        gr.Markdown(f"""
| Metric | Cycle A (Sketch→Photo→Sketch) | Cycle B (Photo→Sketch→Photo) |
|--------|------------------------------|------------------------------|
| SSIM | {ssim_A_arr.mean():.4f} +/- {ssim_A_arr.std():.4f} | {ssim_B_arr.mean():.4f} +/- {ssim_B_arr.std():.4f} |
| PSNR | {psnr_A_arr.mean():.2f}dB | {psnr_B_arr.mean():.2f}dB |

**Architecture**: ResNet Generator ({N_RESBLOCKS} blocks) + PatchGAN Discriminator
**Losses**: LSGAN Adversarial + Cycle Consistency (λ={LAMBDA_CYCLE}) + Identity (λ={LAMBDA_ID})
        """)

demo.launch(share=True, debug=False)
print('Gradio app launched!')

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0daba75f8dc3c76f76.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Gradio app launched!


## Cell 24 — Final Summary

In [24]:
print('=' * 65)
print('  CYCLEGAN FINAL SUMMARY')
print('=' * 65)
print(f'  Config')
print(f'    Image size     : {IMG_SIZE}x{IMG_SIZE}')
print(f'    Batch size     : {BATCH_SIZE}')
print(f'    Epochs         : {NUM_EPOCHS}')
print(f'    ResNet blocks  : {N_RESBLOCKS}')
print(f'    Lambda Cycle   : {LAMBDA_CYCLE}')
print(f'    Lambda Identity: {LAMBDA_ID}')
print(f'    Loss type      : LSGAN (MSE)')
print(f'    D Replay buffer: {BUFFER_SIZE} images')
print(f'    GPUs           : {N_GPUS}')

print(f'\n  Domain A (Sketch): {len(sketch_paths)} images')
print(f'    Sketchy : {len(sketchy_sketch_paths)}')
print(f'    TU-Berlin: {len(tuberlin_sketch_paths)}')
print(f'    QuickDraw: {len(quickdraw_sketch_paths)}')
print(f'  Domain B (Photo) : {len(photo_paths)} images')

print(f'\n  Cycle A (Sketch->Photo->Sketch)')
print(f'    SSIM : {ssim_A_arr.mean():.4f} +/- {ssim_A_arr.std():.4f}')
print(f'    PSNR : {psnr_A_arr.mean():.2f}dB')
print(f'  Cycle B (Photo->Sketch->Photo)')
print(f'    SSIM : {ssim_B_arr.mean():.4f} +/- {ssim_B_arr.std():.4f}')
print(f'    PSNR : {psnr_B_arr.mean():.2f}dB')

print(f'\n  Downloads (Kaggle OUTPUT tab):')
for f in ['/kaggle/working/cyclegan_weights.zip',
          '/kaggle/working/cyclegan_hf_deploy.zip']:
    if os.path.exists(f):
        print(f'    {os.path.basename(f)}  ({os.path.getsize(f)/1e6:.1f} MB)')
print('=' * 65)

  CYCLEGAN FINAL SUMMARY
  Config
    Image size     : 128x128
    Batch size     : 4
    Epochs         : 20
    ResNet blocks  : 6
    Lambda Cycle   : 10.0
    Lambda Identity: 5.0
    Loss type      : LSGAN (MSE)
    D Replay buffer: 50 images
    GPUs           : 1

  Domain A (Sketch): 2000 images
    Sketchy : 0
    TU-Berlin: 2000
    QuickDraw: 0
  Domain B (Photo) : 3000 images

  Cycle A (Sketch->Photo->Sketch)
    SSIM : 0.9404 +/- 0.0212
    PSNR : 27.94dB
  Cycle B (Photo->Sketch->Photo)
    SSIM : 0.7413 +/- 0.0663
    PSNR : 20.52dB

  Downloads (Kaggle OUTPUT tab):
    cyclegan_weights.zip  (68.7 MB)
    cyclegan_hf_deploy.zip  (58.2 MB)
